# 06 — Tempo até o evento: Kaplan–Meier e regressão de Cox

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flavioluizseixas/aprendizado-de-maquina-para-saude/blob/main/notebooks/06_analise_sobrevivencia.ipynb)

**Duração estimada:** 75–90 minutos  
**Pré-requisitos:** Estatística básica e interpretação de intervalos de confiança.

## Objetivos

- distinguir tempo, evento e censura
- estimar Kaplan–Meier e comparar grupos com log-rank
- interpretar hazard ratios de um modelo de Cox
- verificar proporcionalidade e comparar estratificação

## Fonte e licença

[NCCTG Lung Cancer — descrição das variáveis](https://vincentarelbundock.github.io/Rdatasets/doc/survival/lung.html), do pacote R survival, acessada pelo Rdatasets. A documentação explica códigos como status 1=censurado/2=óbito e sexo 1=masculino/2=feminino.

Use os termos do pacote survival/Rdatasets e cite a documentação original.

> **Uso responsável:** Este material tem finalidade exclusivamente educacional. Os resultados não devem ser usados para diagnóstico, prognóstico, tratamento, gestão assistencial ou decisão de saúde pública sem validação adequada, análise de contexto e supervisão de profissionais qualificados.

## Preparação do ambiente

> Como registrar a estratégia de dados ausentes e o código de evento?

In [ ]:
# Preparação reproduzível do ambiente (a instalação ocorre só se faltar pacote).
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO = "flavioluizseixas/aprendizado-de-maquina-para-saude"
REPO_DIR = Path("/content") / REPO.split("/")[-1]
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    command = ["git", "clone", f"https://github.com/{REPO}.git", str(REPO_DIR)]
    if REPO_DIR.exists():
        command = ["git", "-C", str(REPO_DIR), "pull", "--ff-only"]
    subprocess.run(command, check=True)
    os.chdir(REPO_DIR)
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    project = next((p for p in candidates if (p / "src").exists()), Path.cwd())
    os.chdir(project)

packages = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.1,<4', 'matplotlib': 'matplotlib>=3.8,<4', 'seaborn': 'seaborn>=0.13,<1', 'sklearn': 'scikit-learn>=1.4,<2', 'requests': 'requests>=2.31,<3', 'lifelines': 'lifelines>=0.30,<1'}
missing = [spec for module, spec in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from src.config import RANDOM_STATE, seed_everything
seed_everything(RANDOM_STATE)
print(f"Ambiente pronto em {Path.cwd()} | Colab={IN_COLAB} | semente={RANDOM_STATE}")

In [ ]:
from io import StringIO

import matplotlib.pyplot as plt
import pandas as pd
import requests
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.plotting import add_at_risk_counts
from lifelines.statistics import logrank_test

from src.survival_utils import hazard_ratio_table, prepare_lung_data

# No Rdatasets atual, o arquivo canônico usa o alias histórico `cancer`.
# `survival/lung.csv` hoje aponta para outra tabela e é rejeitado pela validação.
URL = "https://vincentarelbundock.github.io/Rdatasets/csv/survival/cancer.csv"

## Pergunta orientadora

> Como o tempo observado e a censura alteram a comparação de sobrevivência entre grupos?

## Obtenção e inspeção

> O evento foi recodificado conforme a fonte: 1=censurado e 2=óbito?

In [ ]:
response = requests.get(URL, timeout=30)
response.raise_for_status()
raw = pd.read_csv(StringIO(response.text))
print("Dimensão original:", raw.shape)
display(raw.head())
display(raw.isna().sum().rename("ausências").to_frame().T)

In [ ]:
lung, preparation = prepare_lung_data(raw, missing="drop")
display(pd.Series(preparation, name="valor").to_frame())
print("Evento=1 significa óbito observado; evento=0 significa censura.")
assert set(lung["event"]) <= {0, 1}

## Kaplan–Meier global

> Qual é a sobrevivência estimada e sua mediana?

In [ ]:
km_global = KaplanMeierFitter(label="Amostra completa")
km_global.fit(lung["time"], event_observed=lung["event"])
ax = km_global.plot_survival_function(ci_show=True)
add_at_risk_counts(km_global, ax=ax)
ax.set(title=f"Kaplan–Meier global (n={len(lung)})", xlabel="Tempo (dias)", ylabel="Sobrevivência estimada")
plt.tight_layout(); plt.show()
print("Mediana de sobrevivência:", km_global.median_survival_time_, "dias")

### Como interpretar

A curva estima a probabilidade de permanecer sem o evento ao longo do tempo, incorporando censura sob pressupostos. A mediana é o tempo em que a estimativa cruza 0,5; não é a expectativa individual.

## Estratificação visual e log-rank

> As curvas por sexo diferem nesta amostra?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
fitted = []
for label, group in lung.groupby("sex_label"):
    km = KaplanMeierFitter(label=label).fit(group["time"], group["event"])
    km.plot_survival_function(ax=ax)
    fitted.append(km)
ax.set(title="Kaplan–Meier por sexo", xlabel="Tempo (dias)", ylabel="Sobrevivência estimada")
add_at_risk_counts(*fitted, ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
male = lung[lung["sex"] == 1]
female = lung[lung["sex"] == 2]
logrank = logrank_test(
    male["time"], female["time"],
    event_observed_A=male["event"], event_observed_B=female["event"],
)
print(f"Log-rank: estatística={logrank.test_statistic:.3f}; p={logrank.p_value:.4f}")

## Cox sem estratificação

> Quais associações permanecem ao considerar idade, sexo e desempenho ECOG?

In [ ]:
cox_data = lung[["time", "event", "age", "sex_female", "ph.ecog"]]
cph = CoxPHFitter()
cph.fit(cox_data, duration_col="time", event_col="event")
display(hazard_ratio_table(cph.summary).round(3))
cph.plot(hazard_ratios=True)
plt.axvline(1, color="grey", linestyle="--")
plt.title("Hazard ratios — Cox sem estratificação"); plt.show()
print("Concordance index:", round(cph.concordance_index_, 3))

In [ ]:
# A função imprime violações e recomendações; não altera o modelo silenciosamente.
cph.check_assumptions(cox_data, p_value_threshold=0.05, show_plots=False)

### Como interpretar

Hazard é uma taxa instantânea condicionada a ainda estar sob risco. HR>1 indica hazard estimado maior por unidade/grupo, não uma diferença de probabilidade absoluta. p-valor não garante relevância clínica nem causalidade.

## Cox estratificado

> O que muda ao permitir uma função de base diferente para cada sexo?

In [ ]:
stratified_data = lung[["time", "event", "age", "ph.ecog", "sex"]]
cph_stratified = CoxPHFitter()
cph_stratified.fit(
    stratified_data, duration_col="time", event_col="event", strata=["sex"]
)
display(hazard_ratio_table(cph_stratified.summary).round(3))
comparison = pd.DataFrame({
    "sem estrato": [cph.concordance_index_, cph.log_likelihood_],
    "sexo como estrato": [cph_stratified.concordance_index_, cph_stratified.log_likelihood_],
}, index=["concordance index", "log-likelihood"])
display(comparison.round(3))

### Como interpretar

O modelo estratificado não produz um único coeficiente para sexo: cada estrato pode ter hazard basal próprio, enquanto idade e ECOG mantêm coeficientes compartilhados. Compare ajuste e pressupostos, não apenas um número.

## Limitações e responsabilidade

- Amostra pequena, observacional e sujeita a confundimento e seleção.
- Censura e o pressuposto de riscos proporcionais precisam ser considerados.
- Hazard ratio não é risco absoluto, diferença de sobrevivência nem efeito causal.

## Atividade

Escolha outra variável documentada, justifique o tratamento de ausências e compare KM/log-rank. Escreva por que significância estatística não basta para relevância clínica.

## Três aprendizados principais

1. Kaplan–Meier incorpora observações censuradas.
2. Cox estima razões de hazards sob pressupostos verificáveis.
3. Estratificar troca um coeficiente comum por hazards basais específicos.

## Referências

- [Rdatasets — lung](https://vincentarelbundock.github.io/Rdatasets/doc/survival/lung.html)
- [R survival — lung](https://stat.ethz.ch/R-manual/R-devel/library/survival/help/lung.html)
- [lifelines documentation](https://lifelines.readthedocs.io/)

## Versões das bibliotecas

Registre o ambiente junto ao resultado.

In [ ]:
from src.config import library_versions
library_versions(('numpy', 'pandas', 'matplotlib', 'lifelines', 'requests'))